In [4]:
# Essential imports for JSON reading
import json
import os
import sys
from typing import List, Dict, Any

print("✅ Essential imports loaded successfully!")

from typing import List, Dict, Any, Optional
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)



# Importing Required Libraries

# Fix numpy compatibility issue with gensim
import warnings
warnings.filterwarnings('ignore', message='.*numpy.dtype size changed.*')
warnings.filterwarnings('ignore', message='.*binary incompatibility.*')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
import os
import csv
import re
from dotenv import load_dotenv
import json 

# Fix VoyageAI API Key Loading
import voyageai
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/code/conf/project_details.env')

# Get API key from environment
voyage_api_key = os.getenv('VOYAGE_API_KEY')
print(f"API Key loaded: {voyage_api_key[:10]}..." if voyage_api_key else "No API key found!")




# Try to import with error handling
try:
    from contor_utils_5 import _read_dictionary
    print("✅ Utils import successful!")
except ImportError as e:
    print(f"❌ Utils import error: {e}")




# Handle gensim import separately to avoid numpy compatibility issues
try:
    # Try to suppress the specific numpy compatibility warning
    import os
    os.environ['PYTHONWARNINGS'] = 'ignore::UserWarning'
    
    from gensim.models import KeyedVectors
    GENSIM_AVAILABLE = True
    print("✅ Gensim import successful!")
except Exception as e:
    print(f"❌ Gensim import error: {e}")
    print("Word embeddings will use fallback mode (zero embeddings).")
    GENSIM_AVAILABLE = False

✅ Essential imports loaded successfully!


INFO:numexpr.utils:NumExpr defaulting to 12 threads.


API Key loaded: pa-ZnppvtJ...
DGL functionality will be limited
✅ Utils import successful!
✅ Gensim import successful!


In [37]:
def import_contor_dataset(path: str, file_name: str, verbose: bool = True) -> List[Dict[str, Any]]:
    """
    Enhanced function to import the CONTOR dataset from a given path and file name.
    
    Args:
        path: The path to the dataset directory
        file_name: Name of the JSON file (e.g., 'train.json', 'dev.json', 'test.json')
        verbose: Whether to print detailed information
    
    Returns:
        List of dictionaries containing the dataset
    """
    file_path = os.path.join(path, file_name)
    
    # Check if file exists
    if not os.path.exists(file_path):
        error_msg = f"File not found: {file_path}"
        logger.error(error_msg)
        raise FileNotFoundError(error_msg)
    
    # Check if file is readable
    if not os.access(file_path, os.R_OK):
        error_msg = f"File not readable: {file_path}"
        logger.error(error_msg)
        raise PermissionError(error_msg)
    
    if verbose:
        print(f"📖 Reading CONTOR dataset from: {file_path}")
        print(f"📊 File size: {os.path.getsize(file_path)} bytes")
    
    data = []
    line_count = 0
    error_count = 0
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                line_count += 1
                
                if not line:  # Skip empty lines
                    continue
                
                try:
                    json_obj = json.loads(line)
                    data.append(json_obj)
                except json.JSONDecodeError as e:
                    error_count += 1
                    error_msg = f"Invalid JSON on line {line_num}: {e}"
                    logger.warning(error_msg)
                    if verbose:
                        print(f"⚠️  {error_msg}")
                        print(f"   Problematic line: {line[:100]}...")
                    continue
                except Exception as e:
                    error_count += 1
                    error_msg = f"Unexpected error on line {line_num}: {e}"
                    logger.error(error_msg)
                    if verbose:
                        print(f"❌ {error_msg}")
                    continue
        
        if verbose:
            print(f"✅ Successfully loaded {len(data)} samples from {file_name}")
            print(f"📈 Total lines processed: {line_count}")
            if error_count > 0:
                print(f"⚠️  Errors encountered: {error_count}")
        
        return data
        
    except UnicodeDecodeError as e:
        error_msg = f"Encoding error reading file {file_path}: {e}"
        logger.error(error_msg)
        if verbose:
            print(f"❌ {error_msg}")
        return []
    except Exception as e:
        error_msg = f"Unexpected error reading file {file_path}: {e}"
        logger.error(error_msg)
        if verbose:
            print(f"❌ {error_msg}")
        return []

def load_contor_transport_dataset(base_path: str = "dataset/contor_datasets/transport/", verbose: bool = True) -> Dict[str, List[Dict[str, Any]]]:
    """
    Enhanced function to load the complete Transport dataset (train, dev, test)
    
    Args:
        base_path: Base path to the transport dataset
        verbose: Whether to print detailed information
        
    Returns:
        Dictionary containing train, dev, and test data
    """
    if verbose:
        print("🚀 Loading CONTOR Transport Dataset")
        print("=" * 50)
    
    dataset = {}
    total_samples = 0
    
    # Define the splits to load
    splits = ['train', 'dev', 'test']
    
    for split in splits:
        if verbose:
            print(f"\n📂 Loading {split} data...")
        
        try:
            dataset[split] = import_contor_dataset(base_path, f'{split}.json', verbose=verbose)
            total_samples += len(dataset[split])
            
            if verbose:
                print(f"✅ {split.capitalize()} data: {len(dataset[split])} samples")
                
        except FileNotFoundError as e:
            if verbose:
                print(f"❌ Error loading {split} data: {e}")
            dataset[split] = []
        except Exception as e:
            if verbose:
                print(f"❌ Unexpected error loading {split} data: {e}")
            dataset[split] = []
    
    if verbose:
        print("\n" + "=" * 50)
        print(f"📊 Dataset Summary:")
        print(f"   Total samples: {total_samples}")
        for split in splits:
            print(f"   {split.capitalize()}: {len(dataset[split])} samples")
    
    return dataset

# Updated analyze_dataset_structure function that returns concepts and labels
def analyze_dataset_structure(dataset: Dict[str, List[Dict[str, Any]]], verbose: bool = True) -> Dict[str, Dict[str, Any]]:
    """
    Analyze the structure of the loaded dataset and return concepts and labels

    Args:
        dataset: The loaded dataset dictionary
        verbose: Whether to print detailed information
        
    Returns:
        Dictionary containing sub-concepts, super-concepts, and labels for each split
    """
    if verbose:
        print("\n🔍 Dataset Structure Analysis")
        print("=" * 50)

    results = {}

    for split_name, split_data in dataset.items():
        if not split_data:
            if verbose:
                print(f"\n📂 {split_name.upper()}: No data")
            results[split_name] = {
                'sub_concepts': set(),
                'super_concepts': set(),
                'labels': set(),
                'count': 0
            }
            continue
            
        if verbose:
            print(f"\n📂 {split_name.upper()}: {len(split_data)} samples")
        
        # Analyze first sample
        sample = split_data[0]
        if verbose:
            print(f"   Sample structure:")
            for key, value in sample.items():
                print(f"     {key}: {type(value).__name__} = {value}")
        
        # Extract concepts and labels
        sub_concepts = set(item.get('v_sub_concept', '') for item in split_data)
        super_concepts = set(item.get('v_super_concept', '') for item in split_data)
        labels = set(item.get('label', '') for item in split_data)
        
        # Store results
        results[split_name] = {
            'sub_concepts': sub_concepts,
            'super_concepts': super_concepts,
            'labels': labels,
            'count': len(split_data)
        }
        
        if verbose:
            print(f"   Unique sub-concepts: {len(sub_concepts)}")
            print(f"   Unique super-concepts: {len(super_concepts)}")
            print(f"   Unique labels: {labels}")

    return results



In [38]:
base_path = "/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport"


import_contor_dataset("/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport" , "train.json")

📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
📊 File size: 331583 bytes
✅ Successfully loaded 1904 samples from train.json
📈 Total lines processed: 1904


[{'v_sub_concept': 'tunnel',
  'v_super_concept': 'stationary artifact',
  'label': 1,
  'rule': 'body=Mid-level-ontology.Tunnel, head=SUMO.StationaryArtifact'},
 {'v_sub_concept': 'pipeline',
  'v_super_concept': 'tunnel',
  'label': 0,
  'rule': 'body=Mid-level-ontology.Pipeline, Mid-level-ontology.Tunnel, head=owl.Bottom'},
 {'v_sub_concept': 'railway junction',
  'v_super_concept': 'transitway junction',
  'label': 1,
  'rule': 'body=transport.RailwayJunction, head=Mid-level-ontology.TransitwayJunction'},
 {'v_sub_concept': 'tunnel',
  'v_super_concept': 'waterway',
  'label': 0,
  'rule': 'body=Mid-level-ontology.Tunnel, Mid-level-ontology.Waterway, head=owl.Bottom'},
 {'v_sub_concept': 'merchant marine',
  'v_super_concept': 'collection',
  'label': 1,
  'rule': 'body=transport.MerchantMarine, head=SUMO.Collection'},
 {'v_sub_concept': 'navigation light',
  'v_super_concept': 'aid to navigation',
  'label': 1,
  'rule': 'body=transport.NavigationLight, head=transport.AidToNavigat

In [39]:
load_contor_transport_dataset(base_path = "/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport")

🚀 Loading CONTOR Transport Dataset

📂 Loading train data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
📊 File size: 331583 bytes
✅ Successfully loaded 1904 samples from train.json
📈 Total lines processed: 1904
✅ Train data: 1904 samples

📂 Loading dev data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/dev.json
📊 File size: 85304 bytes
✅ Successfully loaded 483 samples from dev.json
📈 Total lines processed: 483
✅ Dev data: 483 samples

📂 Loading test data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/test.json
📊 File size: 45096 bytes
✅ Successfully loaded 262 samples from test.json
📈 Total lines processed: 262
✅ Test data: 262 samples

📊 Dataset Summary:
   Total samples: 2649
   Train: 1904 

{'train': [{'v_sub_concept': 'tunnel',
   'v_super_concept': 'stationary artifact',
   'label': 1,
   'rule': 'body=Mid-level-ontology.Tunnel, head=SUMO.StationaryArtifact'},
  {'v_sub_concept': 'pipeline',
   'v_super_concept': 'tunnel',
   'label': 0,
   'rule': 'body=Mid-level-ontology.Pipeline, Mid-level-ontology.Tunnel, head=owl.Bottom'},
  {'v_sub_concept': 'railway junction',
   'v_super_concept': 'transitway junction',
   'label': 1,
   'rule': 'body=transport.RailwayJunction, head=Mid-level-ontology.TransitwayJunction'},
  {'v_sub_concept': 'tunnel',
   'v_super_concept': 'waterway',
   'label': 0,
   'rule': 'body=Mid-level-ontology.Tunnel, Mid-level-ontology.Waterway, head=owl.Bottom'},
  {'v_sub_concept': 'merchant marine',
   'v_super_concept': 'collection',
   'label': 1,
   'rule': 'body=transport.MerchantMarine, head=SUMO.Collection'},
  {'v_sub_concept': 'navigation light',
   'v_super_concept': 'aid to navigation',
   'label': 1,
   'rule': 'body=transport.NavigationL

In [40]:
# Test the updated function
print("🧪 Testing Updated analyze_dataset_structure Function")
print("=" * 60)

# Load the dataset
dataset = load_contor_transport_dataset(base_path = "/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport")

# Analyze and get results
results = analyze_dataset_structure(dataset, verbose=True)

print("\n📊 Returned Results:")
print("=" * 30)

for split_name, split_results in results.items():
    print(f"\n{split_name.upper()}:")
    print(f"  Count: {split_results['count']}")
    print(f"  Sub-concepts: {len(split_results['sub_concepts'])} unique")
    print(f"  Super-concepts: {len(split_results['super_concepts'])} unique")
    print(f"  Labels: {split_results['labels']}")
    
    # Show some examples
    if split_results['sub_concepts']:
        print(f"  Sample sub-concepts: {list(split_results['sub_concepts'])[:5]}")
    if split_results['super_concepts']:
        print(f"  Sample super-concepts: {list(split_results['super_concepts'])[:5]}")

print("\n✅ Function now returns sub-concepts, super-concepts, and labels!")


🧪 Testing Updated analyze_dataset_structure Function
🚀 Loading CONTOR Transport Dataset

📂 Loading train data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
📊 File size: 331583 bytes
✅ Successfully loaded 1904 samples from train.json
📈 Total lines processed: 1904
✅ Train data: 1904 samples

📂 Loading dev data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/dev.json
📊 File size: 85304 bytes
✅ Successfully loaded 483 samples from dev.json
📈 Total lines processed: 483
✅ Dev data: 483 samples

📂 Loading test data...
📖 Reading CONTOR dataset from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/test.json
📊 File size: 45096 bytes
✅ Successfully loaded 262 samples from test.json
📈 Total lines processed: 262
✅ Test data: 262 samples

📊 Da

In [58]:
dataset.keys()

dict_keys(['train', 'dev', 'test'])

In [68]:
results.keys()

dict_keys(['train', 'dev', 'test'])

In [ ]:
#

### VOYAGE Embeddings Testing

In [43]:
# Create a mock KeyedVectors class for fallback
class MockKeyedVectors:
    def load_word2vec_format(self, *args, **kwargs):
        raise NotImplementedError("Gensim not available - use fallback embeddings")

KeyedVectors = MockKeyedVectors


# Define word embedding function locally to avoid import issues
def sep_by_uppercase(str):
    pattern = "[A-Z]"
    new_str = re.sub(pattern, lambda x: " " + x.group(0), str)
    if str[0].islower():  # first character is lowercase
        return new_str
    else:
        return new_str[1:]




# Before preprocessing
concept = "LiquefiedGasTankerShip"

# After preprocessing  
readable_concept = sep_by_uppercase(concept)
# Result: "Liquefied Gas Tanker Ship"




In [44]:
#Voyage Vectors
# Initialize VoyageAI client
vo = voyageai.Client(api_key=voyage_api_key)


documents = ['car' , 'bicycle' , 'Bicycle' , 'CAR' , 'train' , 'Train' , 'Bus' , 'BUS' , 'bus']

# Embed the documents
documents_embeddings = vo.embed(
    documents, model="voyage-3-large", input_type="document"
)


print(documents_embeddings.embeddings)


[[-0.04778236523270607, 0.014563609845936298, 0.00877359788864851, -0.04479034245014191, 0.02125910297036171, 0.013303547166287899, -0.03282225504517555, -0.001605680794455111, -0.028621051460504532, -0.03526311740279198, 0.037209056317806244, 0.026219559833407402, -0.01877325028181076, -0.019791211932897568, 0.02391468919813633, -0.03078632988035679, -0.01643362268805504, 0.023430006578564644, 0.034183286130428314, -0.005013885907828808, -0.0038257951382547617, -0.08211188018321991, -0.00504095247015357, -0.038603831082582474, 0.036714132875204086, -0.027487793937325478, 0.015252561308443546, 0.05543113872408867, 0.018894167616963387, -0.01989244669675827, -0.017653701826930046, -0.024386102333664894, -0.04077473282814026, 0.03396956995129585, -0.023261280730366707, -0.06557702273130417, -0.035679299384355545, -0.018874483183026314, -0.005150270648300648, -0.04701748862862587, -0.015159765258431435, -0.025696519762277603, 0.04775986820459366, -0.10298854112625122, 0.019628113135695457

In [47]:
# Enhanced VoyageAI Word Embedding Function with Cleaned Word List

def voyage_word_embedding(voyage_client, model_name: str, input_type: str, nodes_dict: dict, dim: int = 1024):
    """
    Enhanced function to get VoyageAI embeddings with cleaned word list
    
    Args:
        voyage_client: VoyageAI client instance
        model_name: VoyageAI model name (e.g., "voyage-3.5")
        input_type: Input type for VoyageAI ("document" or "query")
        nodes_dict: Dictionary mapping concept names to indices
        dim: Embedding dimension (default 1024)
    
    Returns:
        numpy array: Embeddings for all nodes
    """
    
    # Initialize embedding matrix
    node_embeding = np.zeros((len(nodes_dict), dim))
    
    # Prepare cleaned list of words for batch processing
    cleaned_words = []
    node_indices = []
    
    print("Processing nodes and cleaning words...")
    
    for nod in nodes_dict:
        # Extract concept name from URI
        words = nod.strip().split('#')
        
        if len(words) == 1:
            # Handle URIs without '#'
            words = words[0].split('/')[-1][:-1]
        else:
            # Handle URIs with '#'
            words = words[1][:-1]
        
        # Convert camelCase to readable text
        cleaned_word = sep_by_uppercase(words)
        
        # Additional cleaning
        cleaned_word = cleaned_word.strip()  # Remove extra spaces
        cleaned_word = ' '.join(cleaned_word.split())  # Remove multiple spaces
        
        # Store cleaned word and its index
        cleaned_words.append(cleaned_word)
        node_indices.append(nodes_dict[nod])
        
        print(f"Original: {nod}")
        print(f"Cleaned: {cleaned_word}")
        print("-" * 40)
    
    # Process in batches to avoid API limits
    batch_size = 10
    total_batches = (len(cleaned_words) + batch_size - 1) // batch_size
    
    print(f"Processing {len(cleaned_words)} words in {total_batches} batches...")
    
    for batch_idx in range(0, len(cleaned_words), batch_size):
        batch_words = cleaned_words[batch_idx:batch_idx + batch_size]
        batch_indices = node_indices[batch_idx:batch_idx + batch_size]
        
        try:
            # Get embeddings for this batch
            result = voyage_client.embed(batch_words, model=model_name, input_type=input_type)
            batch_embeddings = np.array(result.embeddings)
            
            # Store embeddings in the correct positions
            for i, node_idx in enumerate(batch_indices):
                node_embeding[node_idx] = batch_embeddings[i]
            
            print(f"Batch {batch_idx//batch_size + 1}/{total_batches} completed")
            
        except Exception as e:
            print(f"Error processing batch {batch_idx//batch_size + 1}: {e}")
            # Keep zeros for failed batch
    
    print(f"Embedding matrix shape: {node_embeding.shape}")
    print(f"Non-zero embeddings: {np.count_nonzero(node_embeding)}")
    
    return node_embeding

# Test the function
def test_voyage_embedding():
    """Test the enhanced VoyageAI embedding function"""
    
    # Sample nodes dictionary
    test_nodes = {
        "http://example.com/Transport#Car": 0,
        "http://example.com/Vehicle#Truck": 1,
        "http://example.com/Transport#Bicycle": 2,
        "http://example.com/Transport#LiquefiedGasTankerShip": 3,
        "http://example.com/Transport#OffshoreAnchorage": 4
    }
    
    print("Testing VoyageAI embedding function:")
    print("=" * 50)
    
    # Test with VoyageAI client (if available)
    if 'vo' in globals():
        try:
            embeddings = voyage_word_embedding(
                voyage_client=vo,
                model_name="voyage-3.5",
                input_type="document",
                nodes_dict=test_nodes,
                dim=1024
            )
            
            print(f"Success! Embedding shape: {embeddings.shape}")
            print(f"Sample embedding (first 5 values): {embeddings[0][:5]}")
            return embeddings
        except Exception as e:
            print(f"Error: {e}")
    else:
        print("VoyageAI client not available. Run the previous cells first.")

# Run the test
node_embeddings = test_voyage_embedding()


Testing VoyageAI embedding function:
Processing nodes and cleaning words...
Original: http://example.com/Transport#Car
Cleaned: Ca
----------------------------------------
Original: http://example.com/Vehicle#Truck
Cleaned: Truc
----------------------------------------
Original: http://example.com/Transport#Bicycle
Cleaned: Bicycl
----------------------------------------
Original: http://example.com/Transport#LiquefiedGasTankerShip
Cleaned: Liquefied Gas Tanker Shi
----------------------------------------
Original: http://example.com/Transport#OffshoreAnchorage
Cleaned: Offshore Anchorag
----------------------------------------
Processing 5 words in 1 batches...
Batch 1/1 completed
Embedding matrix shape: (5, 1024)
Non-zero embeddings: 5120
Success! Embedding shape: (5, 1024)
Sample embedding (first 5 values): [ 0.01404725 -0.00745406 -0.01257019  0.06754353  0.03651921]


In [46]:
node_embeddings[0].shape

(1024,)

### VOYAGE EMBEDDING CONTOR

In [48]:
def voyage_word_embedding_contor(results: Dict[str, Dict[str, Any]], 
                                  voyage_client=None,
                                  api_key: Optional[str] = None,
                                  model_name: str = "voyage-3-large",
                                  input_type: str = "document",
                                  embedding_dim: int = 1024,
                                  batch_size: int = 128,
                                  cache_file: str = "voyage_embeddings_cache.json",
                                  verbose: bool = True) -> Dict[str, Dict[str, Any]]:
    """
    Generate Voyage AI embeddings for all concepts in the CONTOR results dataset
    and append embeddings to the results.
    
    Args:
        results: Results dictionary from analyze_dataset_structure() with structure:
                 {'train': {'sub_concepts': set(), 'super_concepts': set(), ...}, ...}
        voyage_client: Pre-initialized VoyageAI client (if None, will create one)
        api_key: Voyage API key (if None, will use VOYAGE_API_KEY env var)
        model_name: Voyage model name (default: "voyage-3-large")
        input_type: Input type for embeddings ("document" or "query", default: "document")
        embedding_dim: Embedding dimension (default: 1024)
        batch_size: Batch size for API calls (default: 128, Voyage max is 128)
        cache_file: File to cache embeddings (default: "voyage_embeddings_cache.json")
        verbose: Whether to print progress information
        
    Returns:
        Updated results dictionary with embeddings appended:
        {
            'train': {
                'sub_concepts': set(...),
                'super_concepts': set(...),
                'labels': set(...),
                'count': int,
                'concept_embeddings': dict,  # {concept: numpy array}
                'embedding_matrix': numpy array,  # (num_concepts, embedding_dim)
                'concept_to_index': dict  # {concept: index}
            },
            ...
        }
    """
    import time
    import json as json_lib
    
    if verbose:
        print("\n🚀 Voyage AI Embedding Generation for CONTOR Dataset")
        print("=" * 60)
    
    # Initialize Voyage client if not provided
    if voyage_client is None:
        if api_key is None:
            api_key = os.getenv('VOYAGE_API_KEY')
        if api_key is None:
            raise ValueError("Voyage API key not provided. Set VOYAGE_API_KEY environment variable or pass api_key parameter.")
        voyage_client = voyageai.Client(api_key=api_key)
    
    # Load cache if it exists
    embeddings_cache = {}
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                cached_data = json_lib.load(f)
                # Convert cached embeddings back to numpy arrays
                for concept, emb_list in cached_data.items():
                    embeddings_cache[concept] = np.array(emb_list)
            if verbose:
                print(f"📦 Loaded {len(embeddings_cache)} cached embeddings from {cache_file}")
        except Exception as e:
            if verbose:
                print(f"⚠️  Warning: Could not load cache: {e}")
    
    # Extract all unique concepts from all splits
    all_concepts = set()
    for split_name, split_data in results.items():
        if isinstance(split_data, dict):
            all_concepts.update(split_data.get('sub_concepts', set()))
            all_concepts.update(split_data.get('super_concepts', set()))
    
    # Remove empty strings
    all_concepts = {c for c in all_concepts if c and c.strip()}
    
    if verbose:
        print(f"\n📊 Found {len(all_concepts)} unique concepts across all splits")
    
    # Create concept list for batch processing
    concepts_list = sorted(list(all_concepts))  # Sort for consistency
    concept_to_index = {concept: idx for idx, concept in enumerate(concepts_list)}
    
    # Prepare texts for embedding (concepts are already clean, but we can enhance them)
    texts_to_embed = []
    concept_indices = []
    embeddings_to_get = []
    
    for concept in concepts_list:
        # Check cache first
        if concept in embeddings_cache:
            continue
        
        # Use concept as-is (already clean string)
        # Optionally add context for better embeddings
        enhanced_text = f"Transportation concept: {concept}"
        texts_to_embed.append(enhanced_text)
        concept_indices.append(concept_to_index[concept])
        embeddings_to_get.append(concept)
    
    if verbose:
        print(f"📝 Getting embeddings for {len(texts_to_embed)} concepts ({(len(all_concepts) - len(texts_to_embed))} from cache)")
    
    # Get embeddings in batches
    all_embeddings = np.zeros((len(concepts_list), embedding_dim), dtype=np.float32)
    
    # Fill cached embeddings first
    for concept in concepts_list:
        if concept in embeddings_cache:
            all_embeddings[concept_to_index[concept]] = embeddings_cache[concept]
    
    # Process batches
    num_batches = (len(texts_to_embed) + batch_size - 1) // batch_size
    
    for batch_idx in range(0, len(texts_to_embed), batch_size):
        batch_texts = texts_to_embed[batch_idx:batch_idx + batch_size]
        batch_concepts = embeddings_to_get[batch_idx:batch_idx + batch_size]
        batch_indices = concept_indices[batch_idx:batch_idx + batch_size]
        
        try:
            if verbose:
                print(f"🔄 Processing batch {batch_idx//batch_size + 1}/{num_batches} ({len(batch_texts)} concepts)...")
            
            # Get embeddings from Voyage API
            result = voyage_client.embed(
                batch_texts,
                model=model_name,
                input_type=input_type,
                truncation=True
            )
            
            batch_embeddings = np.array(result.embeddings, dtype=np.float32)
            
            # Store embeddings
            for i, concept in enumerate(batch_concepts):
                idx = batch_indices[i]
                embedding = batch_embeddings[i]
                all_embeddings[idx] = embedding
                embeddings_cache[concept] = embedding
            
            if verbose:
                print(f"   ✅ Batch {batch_idx//batch_size + 1} completed")
            
            # Small delay to respect rate limits
            if batch_idx + batch_size < len(texts_to_embed):
                time.sleep(0.1)
                
        except Exception as e:
            if verbose:
                print(f"   ❌ Error processing batch {batch_idx//batch_size + 1}: {e}")
            # Keep zeros for failed batch (or use cached if available)
    
    # Save cache
    try:
        cache_data = {concept: emb.tolist() for concept, emb in embeddings_cache.items()}
        with open(cache_file, 'w') as f:
            json_lib.dump(cache_data, f)
        if verbose:
            print(f"💾 Saved embeddings cache to {cache_file}")
    except Exception as e:
        if verbose:
            print(f"⚠️  Warning: Could not save cache: {e}")
    
    # Create concept embeddings dictionary (for quick lookup)
    concept_embeddings_dict = {
        concept: all_embeddings[concept_to_index[concept]] 
        for concept in concepts_list
    }
    
    # Append embeddings to results
    updated_results = {}
    for split_name, split_data in results.items():
        updated_results[split_name] = split_data.copy()
        updated_results[split_name]['concept_embeddings'] = concept_embeddings_dict
        updated_results[split_name]['embedding_matrix'] = all_embeddings
        updated_results[split_name]['concept_to_index'] = concept_to_index
        updated_results[split_name]['num_concepts'] = len(concepts_list)
        updated_results[split_name]['embedding_dim'] = embedding_dim
        updated_results[split_name]['model_name'] = model_name
    
    if verbose:
        print(f"\n✅ Embedding generation complete!")
        print(f"   📊 Total concepts: {len(concepts_list)}")
        print(f"   📐 Embedding dimension: {embedding_dim}")
        print(f"   🤖 Model: {model_name}")
        print(f"   📦 Embedding matrix shape: {all_embeddings.shape}")
        print("=" * 60)
    
    return updated_results


In [69]:
results_with_embeddings = voyage_word_embedding_contor(results, voyage_client=vo)


🚀 Voyage AI Embedding Generation for CONTOR Dataset
📦 Loaded 697 cached embeddings from voyage_embeddings_cache.json

📊 Found 697 unique concepts across all splits
📝 Getting embeddings for 0 concepts (697 from cache)
💾 Saved embeddings cache to voyage_embeddings_cache.json

✅ Embedding generation complete!
   📊 Total concepts: 697
   📐 Embedding dimension: 1024
   🤖 Model: voyage-3-large
   📦 Embedding matrix shape: (697, 1024)


In [73]:
results_with_embeddings.keys()

dict_keys(['train', 'dev', 'test'])

In [75]:

results_with_embeddings['train'].keys()

dict_keys(['sub_concepts', 'super_concepts', 'labels', 'count', 'concept_embeddings', 'embedding_matrix', 'concept_to_index', 'num_concepts', 'embedding_dim', 'model_name'])

In [78]:
results_with_embeddings['test'].keys()

dict_keys(['sub_concepts', 'super_concepts', 'labels', 'count', 'concept_embeddings', 'embedding_matrix', 'concept_to_index', 'num_concepts', 'embedding_dim', 'model_name'])

In [80]:
results_with_embeddings['dev'].keys()

dict_keys(['sub_concepts', 'super_concepts', 'labels', 'count', 'concept_embeddings', 'embedding_matrix', 'concept_to_index', 'num_concepts', 'embedding_dim', 'model_name'])

In [77]:

results_with_embeddings['train']['super_concepts']

{'agent',
 'aid to navigation',
 'air route',
 'air traffic control center',
 'air traffic control en route center',
 'air traffic control procedure',
 'air traffic control radar room',
 'air transitway',
 'aircraft',
 'airplane',
 'airport',
 'airport by runway surface',
 'airport classification',
 'anchorage',
 'animal powered device',
 'animal powered vehicle',
 'arriving',
 'artifact',
 'atmospheric region',
 'attaching',
 'automobile',
 'axle',
 'barge',
 'barge carrier ship',
 'battery',
 'bicycle',
 'bill of lading',
 'body motion',
 'boxcar',
 'bridge',
 'broad gauge railway',
 'building',
 'bulk cargo',
 'bulkhead',
 'bus',
 'business railcar',
 'c i a airport length classification',
 'cab car',
 'cable ship',
 'cabotage',
 'canal',
 'canal lock',
 'canal lock gate',
 'canal structure',
 'canoe',
 'car distribution system',
 'cargo container',
 'cargo handling',
 'cargo manifest',
 'cargo ship',
 'catamaran',
 'certificate',
 'certificate of registry',
 'chemical and petroleum

Preprocess Contor Dataset

In [ ]:
import json
import pandas as pd
import numpy as np
import scipy.sparse as sp
from collections import defaultdict

def preprocess_contor_data_old(json_file_path):
    """
    Preprocess CONTOR train.json similar to data_preprocessing_1.ipynb
    """
    
    # 1. Load CONTOR JSON data
    with open(json_file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    # 2. Create nodes dictionary (similar to original)
    nodes_dict = {}
    all_concepts = set()
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        all_concepts.add(sub_concept)
        all_concepts.add(super_concept)
    
    # Assign node IDs
    for i, concept in enumerate(sorted(all_concepts)):
        nodes_dict[concept] = i
    
    # 3. Create label dictionary (template-based)
    label_dict = {}
    label_id = 0
    
    for item in data:
        rule = item['rule']
        if rule not in label_dict:
            label_dict[rule] = label_id
            label_id += 1
    
    # 4. Create label matrix (similar to original)
    num_nodes = len(nodes_dict)
    num_labels = len(label_dict)
    train_labels = sp.lil_matrix((num_nodes, num_labels))
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        rule = item['rule']
        label = item['label']
        
        # Add to label matrix
        if sub_concept in nodes_dict and rule in label_dict:
            node_id = nodes_dict[sub_concept]
            label_id = label_dict[rule]
            train_labels[node_id, label_id] = label
    
    # 5. Create edge list (concept relationships)
    edge_list = []
    relation_dict = {}
    rid = 1
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        label = item['label']
        
        if sub_concept in nodes_dict and super_concept in nodes_dict:
            # Create relationship edge
            relation_type = f"IS_A_{label}"  # Different relation types for valid/invalid
            if relation_type not in relation_dict:
                relation_dict[relation_type] = rid
                rid += 1
            
            src = nodes_dict[sub_concept]
            dst = nodes_dict[super_concept]
            edge_list.append((src, dst, relation_dict[relation_type]))
    
    # 6. Add self-connections (similar to original)
    for node_id in range(num_nodes):
        edge_list.append((node_id, node_id, 0))  # Self-relation
    
    # 7. Convert to numpy array
    edge_list = np.array(edge_list, dtype=np.int32)
    
    return {
        'nodes_dict': nodes_dict,
        'label_dict': label_dict,
        'train_labels': train_labels.tocsr(),
        'edge_list': edge_list,
        'relation_dict': relation_dict
    }

In [60]:
contor_dataset = preprocess_contor_data("/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json")

In [65]:
contor_dataset

{'nodes_dict': {'agent': 0,
  'aid to navigation': 1,
  'air route': 2,
  'air traffic control center': 3,
  'air traffic control en route center': 4,
  'air traffic control procedure': 5,
  'air traffic control radar room': 6,
  'air transitway': 7,
  'aircraft': 8,
  'airplane': 9,
  'airport': 10,
  'airport by runway surface': 11,
  'airport classification': 12,
  'ambulance': 13,
  'anchorage': 14,
  'animal powered device': 15,
  'animal powered vehicle': 16,
  'arriving': 17,
  'artifact': 18,
  'atmospheric region': 19,
  'attaching': 20,
  'automobile': 21,
  'axle': 22,
  'barge': 23,
  'barge carrier ship': 24,
  'battery': 25,
  'bicycle': 26,
  'bill of lading': 27,
  'body motion': 28,
  'boxcar': 29,
  'bridge': 30,
  'broad gauge railway': 31,
  'building': 32,
  'bulk cargo': 33,
  'bulkhead': 34,
  'bus': 35,
  'business railcar': 36,
  'c i a airport length classification': 37,
  'cab car': 38,
  'cable ship': 39,
  'cabotage': 40,
  'canal': 41,
  'canal lock': 42,



🚀 Voyage AI Embedding Generation for CONTOR Dataset
📦 Loaded 384 cached embeddings from voyage_embeddings_cache.json

📊 Found 697 unique concepts across all splits
📝 Getting embeddings for 313 concepts (384 from cache)
🔄 Processing batch 1/3 (128 concepts)...
   ✅ Batch 1 completed
🔄 Processing batch 2/3 (128 concepts)...
   ✅ Batch 2 completed
🔄 Processing batch 3/3 (57 concepts)...
   ✅ Batch 3 completed
💾 Saved embeddings cache to voyage_embeddings_cache.json

✅ Embedding generation complete!
   📊 Total concepts: 697
   📐 Embedding dimension: 1024
   🤖 Model: voyage-3-large
   📦 Embedding matrix shape: (697, 1024)


In [ ]:


def preprocess_contor_data_new(json_file_path, 
                          ftype='voyage', 
                          dim=1024,
                          voyage_embeddings=None,
                          voyage_client=None,
                          voyage_model='voyage-3-large',
                          voyage_input_type='document',
                          bidirectional=False,
                          cache_dir=None,
                          return_dict=True,
                          verbose=True):
    """
    Enhanced preprocess CONTOR train.json with Voyage embeddings as default.
    All functionalities from load_whole_data, optimized for Voyage AI embeddings.
    
    Args:
        json_file_path: Path to CONTOR JSON file (one JSON object per line)
        ftype: Feature type - 'voyage' (default), 'analogy', 'embedding', or 'none'
        dim: Dimension for embeddings (default: 1024 for Voyage-3-large)
        voyage_embeddings: Pre-computed Voyage embeddings dict {concept: embedding_array}
                          If None and ftype='voyage', will generate using voyage_client
        voyage_client: VoyageAI client instance (if None, will try to create from env)
        voyage_model: Voyage model name (default: 'voyage-3-large')
        voyage_input_type: Input type for Voyage ('document' or 'query', default: 'document')
        bidirectional: If True, add reverse edges (bidirectional)
        cache_dir: Directory to cache nodes_dict, label_dict, and features files
        return_dict: If True, return dictionary; if False, return tuple like load_whole_data
        verbose: Whether to print progress information
        
    Returns:
        If return_dict=True:
            Dictionary with all components
        If return_dict=False:
            Tuple: (num_node, edge_list, edge_src, edge_dst, edge_type, edge_norm, 
                   num_rel, node_id_con, labels, node_features)
    """
    
    # 1. Load CONTOR JSON data
    if verbose:
        print(f"📖 Loading CONTOR data from: {json_file_path}")
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    
    if verbose:
        print(f"✅ Loaded {len(data)} samples")
    
    # Determine cache file paths
    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
        node_dict_file = os.path.join(cache_dir, 'all_nodes.dict')
        label_dict_file = os.path.join(cache_dir, 'all_unary_templates.dict')
    else:
        node_dict_file = None
        label_dict_file = None
    
    # 2. Create nodes dictionary with caching
    nodes_dict = None
    if node_dict_file and os.path.exists(node_dict_file):
        if verbose:
            print(f"📦 Loading nodes_dict from cache: {node_dict_file}")
        try:
            nodes_dict = _read_dictionary(node_dict_file)
        except Exception as e:
            if verbose:
                print(f"⚠️  Warning: Could not load cache: {e}")
            nodes_dict = None
    
    if nodes_dict is None:
        if verbose:
            print("📝 Creating nodes_dict from data...")
        nodes_dict = {}
        all_concepts = set()
        
        for item in data:
            sub_concept = item['v_sub_concept']
            super_concept = item['v_super_concept']
            all_concepts.add(sub_concept)
            all_concepts.add(super_concept)
        
        # Assign node IDs
        for i, concept in enumerate(sorted(all_concepts)):
            nodes_dict[concept] = i
        
        # Save to cache
        if node_dict_file:
            if verbose:
                print(f"💾 Saving nodes_dict to cache: {node_dict_file}")
            node_id_str = ''
            for nod_id, nod in enumerate(sorted(all_concepts)):
                node_id_str += str(nod_id) + '\t' + nod + '\n'
            with open(node_dict_file, 'w', encoding='utf-8') as f:
                f.write(node_id_str)
    
    num_nodes = len(nodes_dict)
    if verbose:
        print(f"📊 Number of nodes: {num_nodes}")
    
    # 3. Create relation dictionary with caching
    label_dict = None
    if label_dict_file and os.path.exists(label_dict_file):
        if verbose:
            print(f"📦 Loading label_dict from cache: {label_dict_file}")
        try:
            label_dict = _read_dictionary(label_dict_file)
        except Exception as e:
            if verbose:
                print(f"⚠️  Warning: Could not load cache: {e}")
            label_dict = None
    
    if label_dict is None:
        if verbose:
            print("📝 Creating label_dict from data...")
        label_dict = {}
        label_id = 0
        
        for item in data:

            if item['label'] == 1:

                for idx, template in enumerate(item['rule_template']):
                    rule = template 
                    print(rule)

                    if rule not in label_dict:
                        label_dict[rule] = label_id 
                        label_id += 1

        """
        
        for item in data:

    if item['label'] == 1:

        for idx, template in enumerate(item['rule_template']):
            rule = template 
            print(rule)

            if rule not in label_dict:
                label_dict[rule] = label_id + ((idx) / 10)
            label_id += 1


        """


        # Save to cache
        if label_dict_file:
            if verbose:
                print(f"💾 Saving label_dict to cache: {label_dict_file}")
            label_id_str = ''
            for lab_id, lab in enumerate(sorted(label_dict.keys(), key=lambda x: label_dict[x])):
                label_id_str += str(lab_id) + '\t' + lab + '\n'
            with open(label_dict_file, 'w', encoding='utf-8') as f:
                f.write(label_id_str)
    
    num_labels = len(label_dict)
    if verbose:
        print(f"📊 Number of labels: {num_labels}")
    


    
    # 5. Create edge list (concept relationships)
    if verbose:
        print("📝 Creating edge list...")
    edge_list = []
    relation_dict = {}
    rid = 1  # 0 for self-relation
    node_id_con = set()  # nodes connected with others
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        rule_template1 = item['rule_template'][0]
        rule_template2 = item['rule_template'][1]
        label = item['label']
        
        if sub_concept in nodes_dict and super_concept in nodes_dict and rule_template1 in label_dict and rule_template2 in label_dict:
            
            # Create relationship edge
            relation_type = f"IS_A_{label}"  # Different relation types for valid/invalid

            if relation_type not in relation_dict:

                relation_dict[relation_type] = rid
                rid += 1
            
            src = nodes_dict[sub_concept]
            dst = nodes_dict[super_concept]

            node_id_con.add(src)
            node_id_con.add(dst)
            
            # Not Bidirectional. Src and Destinations are fixed! Only the type of edge is different.
            edge_list.append((src, dst, label_dict[rule_template1]))
            edge_list.append((src, dst, label_dict[rule_template2]))
    
    # 6. Add self-connections (only for connected nodes, like load_whole_data)
    for node_id in node_id_con:
        edge_list.append((node_id, node_id, 0))  # Self-relation
    
    # 7. Sort edges (like load_whole_data)
    edge_list = sorted(edge_list, key=lambda x: (x[1], x[0], x[2]))
    edge_list = np.array(edge_list, dtype=np.int32)
    

    num_nodes = len(nodes_dict)
    num_rel = len(label_dict) + 1  # +1 for self-relation



    # 7.5. Create label matrix
    if verbose:
        print("📝 Creating label matrix...")
    labels = sp.lil_matrix((num_nodes, num_rel))
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        rule1 = item['rule_template'][0]
        rule2 = item['rule_template'][1]
        label = item['label']
        
        if sub_concept in nodes_dict and rule in label_dict:
            node_id1 = nodes_dict[sub_concept]
            node_id2 = nodes_dict[super_concept]
            label_id1 = label_dict[rule1]
            label_id2 = label_dict[rule2]
            labels[node_id1, label_id1] = label
            labels[node_id2, label_id2] = label
    
    labels = labels.tocsr()




    
    # 8. Compute edge normalization (like load_whole_data)
    if verbose:
        print("📝 Computing edge normalization...")
    edge_src, edge_dst, edge_type = edge_list.transpose()
    _, inverse_index, count = np.unique((edge_dst, edge_type), axis=1, 
                                        return_inverse=True, return_counts=True)
    degrees = count[inverse_index]  # c_{i,r} for each relation type
    edge_norm = np.ones(len(edge_dst), dtype=np.float32) / degrees.astype(np.float32)
    
    # 9. Generate node features (Voyage embeddings as default)
    node_features = None
    if ftype != 'none':
        if cache_dir:
            if ftype == 'voyage':
                feature_file = os.path.join(cache_dir, f'all_voyage_features_{dim}.csv')
            elif ftype == 'analogy':
                feature_file = os.path.join(cache_dir, f'all_an_features_{dim}.csv')
            elif ftype == 'embedding':
                feature_file = os.path.join(cache_dir, 'all_em_features.csv')
            else:
                feature_file = None
        else:
            feature_file = None
        
        if feature_file and os.path.exists(feature_file):
            if verbose:
                print(f"📦 Loading node features from cache: {feature_file}")
            try:
                node_features = pd.read_csv(feature_file, sep=',', encoding='utf-8', header=None)
                node_features = node_features.values
                if verbose:
                    print(f"✅ Loaded features with shape: {node_features.shape}")
            except Exception as e:
                if verbose:
                    print(f"⚠️  Warning: Could not load cached features: {e}")
                node_features = None
        
        if node_features is None:
            if verbose:
                print(f"📝 Generating node features (type: {ftype})...")
            
            if ftype == 'voyage':
                # Voyage embeddings (primary/default method)
                if voyage_embeddings:
                    # Use provided Voyage embeddings
                    if verbose:
                        print(f"📦 Using provided Voyage embeddings...")
                    node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                    missing_count = 0
                    for concept, node_id in nodes_dict.items():
                        if concept in voyage_embeddings:
                            emb = voyage_embeddings[concept]
                            # Handle both array and list formats
                            if isinstance(emb, (list, tuple)):
                                emb = np.array(emb)
                            # Truncate or pad to desired dimension
                            if len(emb) >= dim:
                                node_features[node_id] = emb[:dim]
                            else:
                                # Pad with zeros if embedding is shorter
                                node_features[node_id, :len(emb)] = emb
                        else:
                            missing_count += 1
                            if verbose and missing_count <= 5:
                                print(f"⚠️  Warning: No Voyage embedding for concept '{concept}'")
                    if verbose and missing_count > 5:
                        print(f"⚠️  Warning: {missing_count} concepts missing Voyage embeddings")
                
                elif voyage_client:
                    # Generate Voyage embeddings on the fly
                    if verbose:
                        print(f"🚀 Generating Voyage embeddings using {voyage_model}...")
                    try:
                        import voyageai
                        concepts_list = sorted(list(nodes_dict.keys()))
                        texts_to_embed = [f"Transportation concept: {concept}" for concept in concepts_list]
                        
                        # Batch processing
                        batch_size = 128
                        all_embeddings_list = []
                        
                        for i in range(0, len(texts_to_embed), batch_size):
                            batch_texts = texts_to_embed[i:i + batch_size]
                            if verbose:
                                print(f"   Processing batch {i//batch_size + 1}/{(len(texts_to_embed)-1)//batch_size + 1}...")
                            
                            result = voyage_client.embed(
                                batch_texts,
                                model=voyage_model,
                                input_type=voyage_input_type,
                                truncation=True
                            )
                            all_embeddings_list.extend(result.embeddings)
                        
                        # Create feature matrix
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                        for idx, concept in enumerate(concepts_list):
                            emb = np.array(all_embeddings_list[idx])
                            if len(emb) >= dim:
                                node_features[nodes_dict[concept]] = emb[:dim]
                            else:
                                node_features[nodes_dict[concept], :len(emb)] = emb
                        
                        if verbose:
                            print(f"✅ Generated Voyage embeddings with shape: {node_features.shape}")
                    
                    except Exception as e:
                        if verbose:
                            print(f"❌ Error generating Voyage embeddings: {e}")
                            print("⚠️  Falling back to zero embeddings")
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                
                else:
                    # Try to get from environment or create client
                    try:
                        import voyageai
                        voyage_api_key = os.getenv('VOYAGE_API_KEY')
                        if voyage_api_key:
                            if verbose:
                                print(f"🔑 Creating Voyage client from API key...")
                            voyage_client = voyageai.Client(api_key=voyage_api_key)
                            # Recursive call with client
                            # Extract relevant parameters and call again
                            return preprocess_contor_data_new(
                                json_file_path, ftype='voyage', dim=dim,
                                voyage_client=voyage_client, voyage_model=voyage_model,
                                voyage_input_type=voyage_input_type, bidirectional=bidirectional,
                                cache_dir=cache_dir, return_dict=return_dict, verbose=verbose
                            )
                        else:
                            if verbose:
                                print("⚠️  Warning: No Voyage API key found and no embeddings provided")
                            node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                    except ImportError:
                        if verbose:
                            print("⚠️  Warning: voyageai package not available")
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
            
            elif ftype == 'analogy':
                # Use PCA on labels
                labels_dense = labels.todense()
                pca = PCA(n_components=dim)
                node_features = pca.fit_transform(labels_dense)
                if verbose:
                    print(f"✅ Generated analogy features with shape: {node_features.shape}")
            
            elif ftype == 'embedding':
                # Fallback to Word2Vec (if needed)
                if verbose:
                    print("⚠️  Note: Using Word2Vec embeddings (consider using Voyage instead)")
                try:
                    from word_embedding_2 import word_embedding
                    embedding_file = 'dataset/GoogleNews-vectors-negative300.bin.gz'
                    node_features = word_embedding(embedding_file, nodes_dict)
                except ImportError:
                    if verbose:
                        print("⚠️  Warning: word_embedding function not available, using zero embeddings")
                    node_features = np.zeros((num_nodes, dim), dtype=np.float32)
            
            # Save features to cache
            if feature_file and node_features is not None:
                if verbose:
                    print(f"💾 Saving node features to cache: {feature_file}")
                try:
                    with open(feature_file, 'w', encoding='utf-8', newline='') as f:
                        writer = csv.writer(f)
                        # Write without header row for consistency
                        writer.writerows(node_features)
                    if verbose:
                        print(f"✅ Saved features to {feature_file}")
                except Exception as e:
                    if verbose:
                        print(f"⚠️  Warning: Could not save features to cache: {e}")
    
    # Convert labels to dense format (like load_whole_data)
    labels = labels.todense()
    
    if verbose:
        print("✅ Preprocessing complete!")
        print(f"   Nodes: {num_nodes}, Edges: {len(edge_list)}, Relations: {num_rel}")
        if node_features is not None:
            print(f"   Features shape: {node_features.shape}")
    
    # Return format
    if return_dict:
        return {
            'nodes_dict': nodes_dict,
            'label_dict': label_dict,
            'train_labels': labels,
            'labels': labels,
            'edge_list': edge_list,
            'relation_dict': relation_dict,
            'edge_src': edge_src,
            'edge_dst': edge_dst,
            'edge_type': edge_type,
            'edge_norm': edge_norm,
            'num_node': num_nodes,
            'num_rel': num_rel,
            'node_id_con': node_id_con,
            'node_features': node_features
        }
    else:
        # Return tuple format like load_whole_data
        return (num_nodes, edge_list, edge_src, edge_dst, edge_type, edge_norm, 
                num_rel, node_id_con, labels, node_features)

In [6]:


# Custom model and dimensions
result = preprocess_contor_data_new("/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json", voyage_model='voyage-3', dim=1024)

📖 Loading CONTOR data from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
✅ Loaded 1904 samples
📝 Creating nodes_dict from data...
📊 Number of nodes: 618
📝 Creating label_dict from data...
📊 Number of labels: 1904
📝 Creating label matrix...
📝 Creating edge list...
📝 Computing edge normalization...
📝 Generating node features (type: voyage)...
🔑 Creating Voyage client from API key...
📖 Loading CONTOR data from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
✅ Loaded 1904 samples
📝 Creating nodes_dict from data...
📊 Number of nodes: 618
📝 Creating label_dict from data...
📊 Number of labels: 1904
📝 Creating label matrix...
📝 Creating edge list...
📝 Computing edge normalization...
📝 Generating node features (type: voyage)...
🚀 Generating Voyage embeddings using voyage-3...
   Processing batch 1/5...
   Processing batch 2/5...
   Processing batch 3/

In [9]:
result.keys()

dict_keys(['nodes_dict', 'label_dict', 'train_labels', 'labels', 'edge_list', 'relation_dict', 'edge_src', 'edge_dst', 'edge_type', 'edge_norm', 'num_node', 'num_rel', 'node_id_con', 'node_features'])

In [10]:
result['nodes_dict']

{'agent': 0,
 'aid to navigation': 1,
 'air route': 2,
 'air traffic control center': 3,
 'air traffic control en route center': 4,
 'air traffic control procedure': 5,
 'air traffic control radar room': 6,
 'air transitway': 7,
 'aircraft': 8,
 'airplane': 9,
 'airport': 10,
 'airport by runway surface': 11,
 'airport classification': 12,
 'ambulance': 13,
 'anchorage': 14,
 'animal powered device': 15,
 'animal powered vehicle': 16,
 'arriving': 17,
 'artifact': 18,
 'atmospheric region': 19,
 'attaching': 20,
 'automobile': 21,
 'axle': 22,
 'barge': 23,
 'barge carrier ship': 24,
 'battery': 25,
 'bicycle': 26,
 'bill of lading': 27,
 'body motion': 28,
 'boxcar': 29,
 'bridge': 30,
 'broad gauge railway': 31,
 'building': 32,
 'bulk cargo': 33,
 'bulkhead': 34,
 'bus': 35,
 'business railcar': 36,
 'c i a airport length classification': 37,
 'cab car': 38,
 'cable ship': 39,
 'cabotage': 40,
 'canal': 41,
 'canal lock': 42,
 'canal lock gate': 43,
 'canal structure': 44,
 'canoe':

In [11]:
result['label_dict']

{'body=Mid-level-ontology.Tunnel, head=SUMO.StationaryArtifact': 0,
 'body=Mid-level-ontology.Pipeline, Mid-level-ontology.Tunnel, head=owl.Bottom': 1,
 'body=transport.RailwayJunction, head=Mid-level-ontology.TransitwayJunction': 2,
 'body=Mid-level-ontology.Tunnel, Mid-level-ontology.Waterway, head=owl.Bottom': 3,
 'body=transport.MerchantMarine, head=SUMO.Collection': 4,
 'body=transport.NavigationLight, head=transport.AidToNavigation': 5,
 'body=transport.Barge, head=Mid-level-ontology.Watercraft': 6,
 'body=Mid-level-ontology.Bridge, head=SUMO.LandTransitway': 7,
 'body=Mid-level-ontology.Railway, head=SUMO.LandTransitway': 8,
 'body=Mid-level-ontology.AnimalPoweredDevice, transport.TransportationControlDevice, head=owl.Bottom': 9,
 'body=Mid-level-ontology.Waterway, head=SUMO.WaterArea': 10,
 'body=transport.Locomotive, head=transport.RollingStock': 11,
 'body=transport.ChemicalTankerShip, head=transport.CargoShip': 12,
 'body=transport.RailroadBridge, head=Mid-level-ontology.Rai

In [12]:
result['train_labels']

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

In [13]:
result['labels']

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
result['edge_list']

array([[  0,   0,   0],
       [125,   0,   2],
       [153,   0,   2],
       ...,
       [616, 616,   0],
       [107, 617,   2],
       [617, 617,   0]], dtype=int32)

In [15]:
result['relation_dict']

{'IS_A_1': 1, 'IS_A_0': 2}

In [16]:
result['edge_src']

array([  0, 125, 153, ..., 616, 107, 617], dtype=int32)

In [17]:
result['edge_dst']

array([  0,   0,   0, ..., 616, 617, 617], dtype=int32)

In [18]:
result['edge_type']

array([0, 2, 2, ..., 0, 2, 0], dtype=int32)

In [19]:
result['edge_norm']

array([1.   , 0.125, 0.125, ..., 1.   , 1.   , 1.   ], dtype=float32)

In [20]:
result['num_node']

618

In [21]:
result['node_id_con']

{0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [22]:
result['node_features']

array([[ 0.05305956, -0.02155073, -0.01985572, ..., -0.01568919,
         0.02364774,  0.04780785],
       [ 0.02048378, -0.06020527,  0.0143179 , ..., -0.02883693,
         0.00857864,  0.06401259],
       [ 0.03720454, -0.05270103, -0.00284776, ..., -0.03429271,
         0.04349302,  0.06440222],
       ...,
       [ 0.09509031, -0.04880908, -0.02238131, ..., -0.01647557,
         0.01436379,  0.06045818],
       [ 0.08823939, -0.05191094, -0.02483429, ..., -0.00427359,
         0.01690271,  0.05169683],
       [ 0.07342827, -0.075927  , -0.01500777, ..., -0.02427651,
        -0.00142639,  0.03597444]], dtype=float32)